# 🧠 E-Commerce Big Data — Machine Learning & Analytics Pipeline

**Step 5 (Feature Store) & Step 6 (ML Models & Registry)**

Notebook huấn luyện và đánh giá toàn bộ hệ thống Machine Learning trong dự án E-Commerce Big Data:
1. **Feature Store Extraction**: Trích xuất các biến hành vi khách hàng (RFM, tần suất, tỷ lệ chuyển đổi, nhãn churn) từ kho dữ liệu phân tích.
2. **Customer Segmentation (K-Means Clustering)**: Phân cụm khách hàng thành 5 phân khúc kinh doanh (Personas) với Silhouette Score đánh giá chất lượng phân cụm.
3. **Churn Prediction (Random Forest Classifier)**: Dự báo nguy cơ rời bỏ dịch vụ với đầy đủ bộ chỉ số đánh giá (Accuracy, Precision, Recall, F1) và phân tích tầm quan trọng của thuộc tính (Feature Importance).
4. **Product Recommendation Engine**: Thuật toán xếp hạng sản phẩm thịnh hành theo từng danh mục hàng hóa.
5. **Model Registry & Versioning**: Đăng ký và quản lý phiên bản mô hình, lưu vết tham số huấn luyện và metrics vào JSON và bảng cơ sở dữ liệu.

## 1. Setup & Cấu hình Môi trường

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DB_PATH, MODELS_DIR
from src.storage import get_connection, get_database_stats
from src.features import extract_user_features, CLUSTERING_FEATURES, CHURN_FEATURES
from src.ml_models import (
    train_customer_segmentation,
    train_churn_prediction,
    compute_recommendations,
    ModelRegistry
)

print("✅ ML Pipeline modules loaded successfully!")

## 2. Trích xuất Feature Store từ Kho Dữ Liệu (`src.features`)

In [ ]:
# Trích xuất các biến hành vi khách hàng (lấy mẫu 50,000 user để huấn luyện nhanh)
print("Extracting features from event store...")
user_features = extract_user_features(sample_size=50000)

print(f"\nTotal Users Sampled: {len(user_features):,}")
print(f"Overall Churn Rate: {user_features['churned'].mean():.2%}")
display(user_features.head())

## 3. Huấn luyện Mô hình Phân Cụm Khách Hàng (Customer Segmentation — K-Means)

In [ ]:
kmeans_model, scaler, sil_score, seg_summary = train_customer_segmentation(user_features, n_clusters=5)

print(f"==================================================")
print(f"   K-MEANS CLUSTERING (Silhouette Score: {sil_score:.4f})")
print(f"==================================================")
display(seg_summary)

# Trực quan hóa tỷ trọng các phân khúc khách hàng
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=seg_summary, x='persona', y='user_id', palette='Blues_r', ax=ax)
ax.set_title('Customer Count by Behavioral Persona', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Users')
ax.set_xlabel('Persona')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 4. Huấn luyện Mô hình Dự Báo Rời Bỏ (Churn Prediction — Random Forest)

In [ ]:
rf_model, metrics, importance_df = train_churn_prediction(extract_user_features(sample_size=50000, for_churn=True))

print(f"==================================================")
print(f"   RANDOM FOREST CHURN CLASSIFIER EVALUATION")
print(f"==================================================")
for k, v in metrics.items():
    print(f"  {k.upper():<12}: {v:.4f}")

# Trực quan hóa Feature Importance
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=importance_df, x='importance', y='feature', palette='viridis', ax=ax)
ax.set_title('Feature Importance for Customer Churn Prediction', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 5. Xây dựng Danh mục Gợi ý Sản phẩm (Product Recommendation Engine)

In [ ]:
recs_df = compute_recommendations(top_n_per_category=5)
print(f"Top Recommendations Catalog ({len(recs_df)} items):")
display(recs_df.head(15))

## 6. Kiểm tra Sổ đăng ký Mô hình (Model Registry & Versioning)

In [ ]:
registry_path = MODELS_DIR / 'model_registry.json'
with open(registry_path, 'r', encoding='utf-8') as f:
    reg_data = json.load(f)

print("Model Registry Records (JSON):")
print(json.dumps(reg_data, indent=2))

# Kiểm tra dữ liệu được ghi trong SQLite table `model_performance`
conn = get_connection()
model_perf_df = pd.read_sql_query("SELECT * FROM model_performance;", conn)
conn.close()

print("\nModel Performance Table in SQLite:")
display(model_perf_df)